### Structured outputs


In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

### Pydantic
pydentic models provide the richest feature set with field validation,discriptions and nested structures

In [8]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field( description="The title of the movie")
    year: int = Field( description="The release year of the movie")
    genre: str = Field( description="The genre of the movie")
    director: str = Field( description="The director of the movie")
    rating: float = Field( description="The rating of the movie")

In [9]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025E9A5902F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025E9A590D70>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [14]:
model_with_structure.invoke("Tell me about the movie Inside Out")

Movie(title='Inside Out', year=2015, genre='Animation', director='Pete Docter', rating=8.1)

In [15]:
model.invoke("Tell me about the movie Inside Out")


AIMessage(content='<think>\nOkay, so I need to explain the movie Inside Out. Let me start by recalling what I know. It\'s an animated film from Pixar, right? I think it came out in 2015. The main characters are emotions living inside a girl named Riley. Her name is Riley Andersen. The story follows her emotions as they navigate her life. The main ones are Joy, Sadness, Anger, Fear, and Disgust. They\'re in her mind controlling her actions. \n\nThe plot probably revolves around Riley moving to a new city with her parents, which causes her to feel overwhelmed. The emotions have to deal with this big change. I remember that Joy is the main one trying to keep things positive, but there\'s a conflict with Sadness. Maybe they get separated, and the rest have to manage on their own. \n\nThere\'s a scene where Joy and Sadness get sucked out of Riley\'s mind and end up in the memory dumps. They have to find their way back while the other emotions are stuck running Riley\'s life without them. Th

In [17]:
response = model_with_structure.invoke("Provide details about the moview Inside Out")
response

Movie(title='Inside Out', year=2015, genre='Animation, Family, Adventure', director='Pete Docter, Ronnie Del Carmen', rating=8.2)

### Message output alongside parsed structure

In [18]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details. """
    title: str = Field( ... , description="The title of the movie")
    year: int = Field( ... , description="The year the movie was released")
    director: str = Field( ... , description="The director of the movie")
    rating: float = Field( ... , description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the functions I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception. The title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb gives it an 8.8/10. So I have all the required parameters. I\'ll structure the tool call with those details. Let me double-check the function\'s required fields again to make sure I didn\'t miss anything. Yep, title, year, director, and rating are all required. Alright, I\'m ready to format the response.\n', 'tool_calls': [{'id': '8j5ccf899', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]},

### Nested Structure


In [20]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")
model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie The Matrix")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie The Matrix. Let me see what tools I have available. There\'s a function called MovieDetails that requires title, year, cast, and genres. The parameters also include budget, which is optional.\n\nFirst, I need to recall information about The Matrix. The title is straightforward. The release year was 1999. The main cast includes Keanu Reeves as Neo, Laurence Fishburne as Morpheus, Carrie-Anne Moss as Trinity, and Hugo Weaving as Agent Smith. The genres are Action, Sci-Fi, and Thriller. The budget was around $63 million. \n\nI should structure the response using the MovieDetails function. Let me check the required fields again: title, year, cast, genres. Budget is optional but adding it might be helpful. Each cast member should be an object with name and role. Genres are an array of strings. \n\nWait, the parameters for cast are an array of objects with name and role

### TypedDict


In [26]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details. """
    title: Annotated[str, ... , "The title of the movie"]
    year: Annotated[int, ... , "The year the movie was released"]
    director: Annotated[str, ... , "The director of the movie"]
    rating: Annotated[float, ... , "The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
model_withtypedict.invoke("Please provide the details of the movie 3 idiots")

{'director': 'Rajkumar Hirani',
 'rating': 8.4,
 'title': '3 Idiots',
 'year': 2009}

In [27]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")
model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie The Matrix")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "The Matrix." I need to use the MovieDetails function to get the information. Let me check the parameters required for that function. The function needs title, year, cast, genres, and budget.\n\nFirst, the title is obviously "The Matrix." The year it was released was 1999. For the cast, I should include the main actors and their roles. Keanu Reeves as Neo, Laurence Fishburne as Morpheus, Carrie-Anne Moss as Trinity, and Hugo Weaving as Agent Smith. Then there\'s the genres: Action, Sci-Fi. The budget was around $63 million.\n\nWait, let me make sure about the budget figure. I think it\'s correct, but maybe I should double-check. Yeah, I recall that The Matrix had a budget of about $63 million. Okay, putting it all together into the function parameters. Each cast member needs to be a dictionary with name and role. Genres as an array of strings. Make sure all requir